01. Setup

In [1]:
# Instalação do Java e Spark
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# Variáveis de ambiente
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Pacote Iceberg
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.0-bin-hadoop3/jars/

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Sessão Spark
spark = SparkSession.builder \
.appName("IcebergWithSpark") \
.config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
.config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
.config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
.config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
.config("spark.sql.default.catalog", "hadoop_catalog") \
.getOrCreate()

# Diretório do Warehouse
!mkdir -p /content/warehouse

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,951 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,945 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,011 kB]
Get:14 http://security.ubunt

In [27]:
from pyspark.sql.functions import to_date, col

# Partição de Dados

In [28]:
# Verifica tabela e exclui se existir
spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.vendas_partitioned")

# Cria tabela particionada Iceberg
spark.sql("""
    CREATE TABLE hadoop_catalog.default.vendas_partitioned (
        id INT,
        produto STRING,
        quantidade INT,
        preco DOUBLE,
        data_venda DATE,
        categoria STRING
    )
    USING iceberg
    PARTITIONED BY (categoria, days(data_venda))
""")

DataFrame[]

In [29]:
# inserimos vendas
data = [
    (1, "Produto A", 10, 15.5, "2023-11-01", "Eletrônicos"),
    (2, "Produto B", 5, 22.0, "2023-11-02", "Vestuário"),
    (3, "Produto C", 8, 30.0, "2023-11-03", "Eletrônicos"),
    (4, "Produto D", 12, 25.0, "2023-11-04", "Alimentos"),
    (5, "Produto E", 7, 18.5, "2023-11-05", "Vestuário"),
    (6, "Produto F", 9, 20.0, "2023-11-06", "Alimentos"),
    (7, "Produto G", 15, 35.0, "2023-11-07", "Eletrônicos"),
    (8, "Produto H", 1, 35.0, "2023-11-07", "Eletrônicos")
]
columns = ["id", "produto", "quantidade", "preco", "data_venda", "categoria"]

df = spark.createDataFrame(data, columns)
df = df.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))

df.writeTo("hadoop_catalog.default.vendas_partitioned").append()

In [30]:
# Listamos as Partições
partitions_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas_partitioned.partitions")
partitions_df.show(truncate=False)

+-------------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition                |spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+-------------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{Eletrônicos, 2023-11-01}|0      |1           |1         |1731                         |0                           |0                         |0                           |0                         |2026-03-30 20:34:02.112|4475761134997898436   

In [31]:
# Deve excluir registro de número 5, partição  Vestuário, 2023-11-05
spark.sql("""
    DELETE FROM hadoop_catalog.default.vendas_partitioned
    WHERE categoria = 'Vestuário' AND data_venda = '2023-11-05'
""")

DataFrame[]

In [32]:
# Partições Atualizadas
partitions_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas_partitioned.partitions")
partitions_df.show(truncate=False)

+-------------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition                |spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+-------------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{Eletrônicos, 2023-11-01}|0      |1           |1         |1731                         |0                           |0                         |0                           |0                         |2026-03-30 20:34:02.112|4475761134997898436   

In [33]:
# Categoria Eletrônicos em 7 / 11
filtered_df = spark.sql("""
    SELECT * FROM hadoop_catalog.default.vendas_partitioned
    WHERE categoria = 'Eletrônicos' AND data_venda = '2023-11-07'
""")
filtered_df.show()

+---+---------+----------+-----+----------+-----------+
| id|  produto|quantidade|preco|data_venda|  categoria|
+---+---------+----------+-----+----------+-----------+
|  7|Produto G|        15| 35.0|2023-11-07|Eletrônicos|
|  8|Produto H|         1| 35.0|2023-11-07|Eletrônicos|
+---+---------+----------+-----+----------+-----------+



In [34]:
# Vamos criar uma tabela clientes. Primeiro verificamos se existe
spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.clientes_partitioned")

# Partição por cidade - categoria preferida deve ser as existentes em vendas
spark.sql("""
    CREATE TABLE hadoop_catalog.default.clientes_partitioned (
        cliente_id INT,
        nome STRING,
        cidade STRING,
        categoria_preferida STRING
    )
    USING iceberg
    PARTITIONED BY (cidade)
""")

DataFrame[]

In [35]:
# Dados
customer_data = [
    (1, "Alice", "São Paulo", "Eletrônicos"),
    (2, "Bob", "Rio de Janeiro", "Vestuário"),
    (3, "Carol", "São Paulo", "Alimentos"),
    (4, "David", "Porto Alegre", "Eletrônicos"),
    (5, "Eve", "Novo Hamburgo", "Vestuário")
]
customer_columns = ["cliente_id", "nome", "cidade", "categoria_preferida"]

df_customers = spark.createDataFrame(customer_data, customer_columns)
df_customers.writeTo("hadoop_catalog.default.clientes_partitioned").append()

In [36]:
# Join vendas e clientes pela categoria
join_df = spark.sql("""
    SELECT v.id, v.produto, v.quantidade, v.preco, v.data_venda, v.categoria,
           c.cliente_id, c.nome, c.cidade
    FROM hadoop_catalog.default.vendas_partitioned v
    INNER JOIN hadoop_catalog.default.clientes_partitioned c
    ON v.categoria = c.categoria_preferida
""")
join_df.show()

+---+---------+----------+-----+----------+-----------+----------+-----+--------------+
| id|  produto|quantidade|preco|data_venda|  categoria|cliente_id| nome|        cidade|
+---+---------+----------+-----+----------+-----------+----------+-----+--------------+
|  1|Produto A|        10| 15.5|2023-11-01|Eletrônicos|         4|David|  Porto Alegre|
|  1|Produto A|        10| 15.5|2023-11-01|Eletrônicos|         1|Alice|     São Paulo|
|  3|Produto C|         8| 30.0|2023-11-03|Eletrônicos|         4|David|  Porto Alegre|
|  3|Produto C|         8| 30.0|2023-11-03|Eletrônicos|         1|Alice|     São Paulo|
|  4|Produto D|        12| 25.0|2023-11-04|  Alimentos|         3|Carol|     São Paulo|
|  2|Produto B|         5| 22.0|2023-11-02|  Vestuário|         2|  Bob|Rio de Janeiro|
|  2|Produto B|         5| 22.0|2023-11-02|  Vestuário|         5|  Eve| Novo Hamburgo|
|  7|Produto G|        15| 35.0|2023-11-07|Eletrônicos|         4|David|  Porto Alegre|
|  7|Produto G|        15| 35.0|

In [37]:
# Join com filtro baseado em cidade e data
filtered_join_df = spark.sql("""
    SELECT v.id, v.produto, v.quantidade, v.preco, v.data_venda, v.categoria,
           c.cliente_id, c.nome, c.cidade
    FROM hadoop_catalog.default.vendas_partitioned v
    INNER JOIN hadoop_catalog.default.clientes_partitioned c
    ON v.categoria = c.categoria_preferida
    WHERE c.cidade = 'São Paulo' AND v.data_venda >= '2023-11-03'
""")
filtered_join_df.show()

+---+---------+----------+-----+----------+-----------+----------+-----+---------+
| id|  produto|quantidade|preco|data_venda|  categoria|cliente_id| nome|   cidade|
+---+---------+----------+-----+----------+-----------+----------+-----+---------+
|  3|Produto C|         8| 30.0|2023-11-03|Eletrônicos|         1|Alice|São Paulo|
|  4|Produto D|        12| 25.0|2023-11-04|  Alimentos|         3|Carol|São Paulo|
|  7|Produto G|        15| 35.0|2023-11-07|Eletrônicos|         1|Alice|São Paulo|
|  8|Produto H|         1| 35.0|2023-11-07|Eletrônicos|         1|Alice|São Paulo|
|  6|Produto F|         9| 20.0|2023-11-06|  Alimentos|         3|Carol|São Paulo|
+---+---------+----------+-----+----------+-----------+----------+-----+---------+

